Cell 1 — Install

In [1]:
!pip install -U anthropic python-dotenv pandas tabulate


  Attempting uninstall: anthropic
    Found existing installation: anthropic 0.66.0
    Uninstalling anthropic-0.66.0:
      Successfully uninstalled anthropic-0.66.0


Cell 2 — Load API key

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("keys.env")  # or ".env"
anthropic_key = os.getenv("ANTHROPIC_API_KEY")
assert anthropic_key, "Missing ANTHROPIC_API_KEY in keys.env/.env"
print("ANTHROPIC_API_KEY loaded (masked):", anthropic_key[:6] + "..." + anthropic_key[-4:])


AssertionError: Missing ANTHROPIC_API_KEY in keys.env/.env

Cell 3 — Imports & config

In [3]:
import time, math, pandas as pd
from tabulate import tabulate
from typing import List, Dict, Any
import anthropic

PROMPT = "Explain transformers in AI in 3 short sentences."
N_RUNS = 3
MAX_OUTPUT_TOKENS = 128
TEMPERATURE = 0.5

client = anthropic.Anthropic(api_key=anthropic_key)


Cell 4 — Core runner (streaming)

In [5]:
def run_once_anthropic(model_id: str, prompt: str,
                       max_output_tokens: int = MAX_OUTPUT_TOKENS,
                       temperature: float = TEMPERATURE) -> Dict[str, Any]:
    """
    Correct streaming pattern for anthropic>=0.34:
      - iterate stream.text_stream
      - call stream.get_final_message() at the end
    Measures:
      - ttfb_s
      - time_to_final_token_s
      - total_s
    """
    t0 = time.perf_counter()
    first = None
    last = None
    parts = []

    with client.messages.stream(
        model=model_id,
        max_tokens=max_output_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for delta in stream.text_stream:
            now = time.perf_counter()
            if first is None:
                first = now
            last = now
            parts.append(delta)

        # ensure full message has been received/closed
        _final = stream.get_final_message()

    total_s = time.perf_counter() - t0
    ttfb_s = (first - t0) if first else math.nan
    t2last_s = (last - first) if (first and last) else math.nan

    return {
        "model": model_id,
        "ttfb_s": ttfb_s,
        "time_to_final_token_s": t2last_s,
        "total_s": total_s,
        "text": "".join(parts),
    }


Cell 5 — Benchmark function

In [7]:
def bench_models_anthropic(models: List[str], prompt: str,
                           runs: int = N_RUNS,
                           csv_path: str = None) -> pd.DataFrame:
    rows = []
    for mid in models:
        print(f"\n--- {mid} ---")
        try:
            # Warm-up
            _ = run_once_anthropic(mid, prompt)

            # Measured runs
            times = [run_once_anthropic(mid, prompt) for _ in range(runs)]
            ttfb = [t["ttfb_s"] for t in times if not math.isnan(t["ttfb_s"])]
            t2last = [t["time_to_final_token_s"] for t in times if not math.isnan(t["time_to_final_token_s"])]
            total = [t["total_s"] for t in times]

            def summary(xs): return (min(xs), sum(xs)/len(xs), max(xs))

            ttfb_stats = summary(ttfb) if ttfb else (math.nan, math.nan, math.nan)
            t2last_stats = summary(t2last) if t2last else (math.nan, math.nan, math.nan)
            total_stats = summary(total)

            print("TTFB          min/avg/max:", ttfb_stats)
            print("Time-to-final min/avg/max:", t2last_stats)
            print("Total         min/avg/max:", total_stats)

            rows.append({
                "model": mid,
                "ttfb_min": ttfb_stats[0], "ttfb_avg": ttfb_stats[1], "ttfb_max": ttfb_stats[2],
                "t2last_min": t2last_stats[0], "t2last_avg": t2last_stats[1], "t2last_max": t2last_stats[2],
                "total_min": total_stats[0], "total_avg": total_stats[1], "total_max": total_stats[2],
            })

        except Exception as e:
            print("Error:", e)
            rows.append({
                "model": mid,
                "ttfb_min": None, "ttfb_avg": None, "ttfb_max": None,
                "t2last_min": None, "t2last_avg": None, "t2last_max": None,
                "total_min": None, "total_avg": None, "total_max": None,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("total_avg", na_position="last")

    if csv_path:
        df.to_csv(csv_path, index=False)
        print(f"\nSaved: {csv_path}")

    if not df.empty:
        print("\n=== Sorted by total_avg (lower is faster) ===")
        print(tabulate(df, headers="keys", tablefmt="github", floatfmt=".3f"))
    return df


Cell 6 — Run Anthropic benchmark

In [ ]:
# Popular Claude models (Sept 2025, adjust if you see different names in your account)
anthropic_models = [
    "claude-3-5-sonnet-20240620",
    "claude-3-5-haiku-20241022",
    "claude-3-opus-20240229",   # slower, reasoning-heavy
]

df_anthropic = bench_models_anthropic(anthropic_models, PROMPT, runs=3,
                                      csv_path="anthropic_latency_baseline.csv")



--- claude-3-5-sonnet-20240620 ---
Error: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

--- claude-3-5-haiku-20241022 ---
Error: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

--- claude-3-opus-20240229 ---
Error: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

Saved: anthropic_latency_baseline.csv

=== Sorted by total_avg (lower is faster) ===
|    | model                      | ttfb_min   | ttfb_avg   | ttfb_max   | t2last_min   | t2last_avg   | t2last_max   | total_min   | total_avg   | total_max   |
|----|----------------------------|------------|------------|------------|--------------|--------------|--

C:\Users\nirosh\AppData\Local\Temp\ipykernel_16832\4100293513.py:9: DeprecationWarning: The model 'claude-3-5-sonnet-20240620' is deprecated and will reach end-of-life on October 22, 2025.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  _ = run_once_anthropic(mid, prompt)
C:\Users\nirosh\AppData\Local\Temp\ipykernel_16832\4100293513.py:9: DeprecationWarning: The model 'claude-3-opus-20240229' is deprecated and will reach end-of-life on January 5th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  _ = run_once_anthropic(mid, prompt)


: 